# **IBM Data Science Professional Certificate <br>Capstone Project**

## Building an interactive dash application 

The dashboard includes a pie chart that displays the share of successful launches for all sites, with a dropdown menu to select a specific site for more detailed insights. 

Also, it includes an scatter plot showing the relationship between payload, launch outcome and booster version categories. An slider allows users to filter the data by payload mass range.

In [ ]:
# Install dash if necessary
# !pip install dash

In [ ]:
# Import required libraries
import pandas as pd
import dash
from dash import Dash, html, dcc
from dash.dependencies import Input, Output
import plotly.express as px

In [ ]:
# Read the spacex data into pandas dataframe
spacex_df = pd.read_csv("../data/spacex_launch_dash.csv")
# csv file contains the data to include in dash application: Successful Launches by Site and Payload Mass (kg)
max_payload = spacex_df['Payload Mass (kg)'].max()
min_payload = spacex_df['Payload Mass (kg)'].min()

# Create a dash application
app = dash.Dash(__name__)

# Create an app layout
app.layout = html.Div(children=[html.H1('SpaceX Launch Records Dashboard',
                                        style={'textAlign': 'center', 'color': '#503D36',
                                               'font-size': 40}),
            # Add a dropdown list to enable Launch Site selection
            # The default select value is for ALL sites
            # dcc.Dropdown(id='site-dropdown',...)
            dcc.Dropdown(
            id='site-dropdown',
            options=[
                {'label': 'All Sites', 'value': 'ALL'},
                {'label': 'CCAFS LC-40', 'value': 'CCAFS LC-40'}, # name of launch site
                {'label': 'CCAFS SLC-40', 'value': 'CCAFS SLC-40'}, # name of launch site
                {'label': 'KSC LC-39A', 'value': 'KSC LC-39A'}, # name of launch site
                {'label': 'VAFB SLC-4E', 'value': 'VAFB SLC-4E'} # name of launch site
            ],
        value='ALL',
        placeholder="Select a Launch Site",
        searchable=True
        ),

        html.Br(),

        # Add a pie chart to show the total successful launches count for all sites
        # If a specific launch site was selected, show the Successful vs. Failed counts for the site
        html.Div(dcc.Graph(id='success-pie-chart')),
        html.Br(),

        html.P("Payload range (Kg):"),
        # Add a slider to select payload range
        #dcc.RangeSlider(id='payload-slider',...)
        dcc.RangeSlider(id='payload-slider',
                            min=0,
                            max=10000,
                            step=1000,
                            value=[min_payload, max_payload]
                            ),

        # Add a scatter chart to show the correlation between payload and launch success
        html.Div(dcc.Graph(id='success-payload-scatter-chart')),
                ])

# Add a callback function for `site-dropdown` as input, `success-pie-chart` as output
@app.callback(Output(component_id='success-pie-chart', component_property='figure'),
              Input(component_id='site-dropdown', component_property='value'))
def get_pie_chart(entered_site):
    filtered_df = spacex_df
    if entered_site == 'ALL':
        fig = px.pie(filtered_df, values='class', 
        names='Launch Site', 
        title='Total Success Launches by Site')
        return fig
    else:
        # return the outcomes piechart for a selected site
        df = spacex_df[spacex_df['Launch Site'] == entered_site]
        filtered_df = df.groupby('class').size().reset_index(name='class count')
        fig = px.pie(filtered_df, values='class count',
        names='class',
        title='Total Success Launches for Site %s' % entered_site)
        return fig

# Add a callback function for `site-dropdown` and `payload-slider` as inputs, `success-payload-scatter-chart` as output
@app.callback(Output(component_id='success-payload-scatter-chart',component_property='figure'),
                [Input(component_id='site-dropdown',component_property='value'),
                Input(component_id='payload-slider',component_property='value')])
def scatter(entered_site,payload):
    filtered_df = spacex_df[spacex_df['Payload Mass (kg)'].between(payload[0],payload[1])]
    if entered_site=='ALL':
        fig=px.scatter(filtered_df,x='Payload Mass (kg)',y='class',color='Booster Version Category',title='Correlation between Payload and Success for all Sites')
        return fig
    else:
        fig=px.scatter(filtered_df[filtered_df['Launch Site']==entered_site],x='Payload Mass (kg)',y='class',color='Booster Version Category',title=f"Correlation between Payload and Success for site {entered_site}")
        return fig


# Run the app
if __name__ == '__main__':
    app.run(debug=False)